# Extração de categorias — Drogaria Araujo

Fonte: <https://www.araujo.com.br/>

O site roda em **Salesforce Commerce Cloud** e fica atrás do WAF da Akamai: requisição com `requests`
ou com Playwright *headless* devolve **403 Access Denied**. Por isso o download é feito com o
Chromium **visível** (`headless=False`), que passa pela verificação.

O menu principal (`ul.mainMenu__categories`) já vem no HTML da home com a árvore completa de
3 níveis — não é preciso clicar em nada nem visitar cada departamento.

## 1. Setup

Rode uma vez só (o kernel precisa ser o `.venv` de `ExtracaoDeDados`).

In [1]:
# %pip install playwright pandas beautifulsoup4
# !playwright install chromium

from playwright.async_api import async_playwright
from bs4 import BeautifulSoup
import pandas as pd

BASE = "https://www.araujo.com.br"
UA = ("Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
      "(KHTML, like Gecko) Chrome/151.0.0.0 Safari/537.36")

## 2. Baixar o HTML da home

Dentro do notebook o kernel já roda um *event loop* do asyncio, então a **API sync** do Playwright
quebra com `It looks like you are using Playwright Sync API inside the asyncio loop`.
Em notebook usa-se a **API async** com `await` direto na célula (top-level await).

Uma janela do Chromium vai abrir e fechar sozinha.

In [2]:
async def baixar_home() -> str:
    async with async_playwright() as p:
        navegador = await p.chromium.launch(headless=False)   # headless=True -> 403
        contexto = await navegador.new_context(
            user_agent=UA,
            locale="pt-BR",
            timezone_id="America/Sao_Paulo",
            viewport={"width": 1440, "height": 900},
        )
        pagina = await contexto.new_page()
        await pagina.goto(BASE, wait_until="domcontentloaded", timeout=60000)
        # o menu vem no HTML mas fica escondido (abre no hover) -> state="attached"
        await pagina.wait_for_selector("ul.mainMenu__categories li a", state="attached", timeout=30000)
        html = await pagina.content()
        await navegador.close()
    return html

html = await baixar_home()

with open("home-araujo.html", "w", encoding="utf-8") as f:
    f.write(html)

print(f"HTML baixado: {len(html):,} caracteres")

HTML baixado: 1,463,905 caracteres


## 3. Percorrer a árvore do menu

A marcação do site é meio torta (`<ul>` com `<div>` no meio antes dos `<li>`), então a função
`filhos()` pega os `<li>` nos dois formatos. `anda()` é recursiva e guarda a trilha do caminho.

In [3]:
def filhos(ul):
    """<li> filhos diretos, com ou sem <div> no meio."""
    return ul.select(":scope > li") + ul.select(":scope > div > li")


def nome_do_link(a):
    return a.get("title") or " ".join(a.get_text(" ", strip=True).split())


def anda(ul, nivel, trilha, linhas):
    for li in filhos(ul):
        a = li.find("a", recursive=False)
        if not a or not a.get("href"):
            continue

        nome = nome_do_link(a)
        href = a["href"]

        linhas.append({
            "nivel": nivel,
            "categoria": nome,
            "categoria_pai": trilha[-1] if trilha else None,
            "caminho": " > ".join(trilha + [nome]),
            "url": BASE + href if href.startswith("/") else href,
            "slug": href.strip("/").split("/")[-1],
        })

        subcategorias = li.find("ul", recursive=False)
        if subcategorias:
            anda(subcategorias, nivel + 1, trilha + [nome], linhas)


sopa = BeautifulSoup(html, "html.parser")
menu = sopa.select_one("ul.mainMenu__categories")

linhas = []
anda(menu, nivel=1, trilha=[], linhas=linhas)

df = pd.DataFrame(linhas)
print(f"{len(df)} categorias extraídas")
df.head(10)

750 categorias extraídas

,nivel,categoria,categoria_pai,caminho,url,slug
0,1,Medicamentos,NaN,Medicamentos,https://www.araujo.com.br/medicamentos,medicamentos
1,2,Remédio para Diabetes,Medicamentos,Medicamentos > Remédio para Diabetes,https://www.araujo.com.br/medicamentos/diabetes,diabetes
2,3,Insulina,Remédio para Diabetes,Medicamentos > Remédio para Diabetes > Insulina,https://www.araujo.com.br/medicamentos/insulina,insulina
3,2,Remédios Cardiológicos,Medicamentos,Medicamentos > Remédios Cardiológicos,https://www.araujo.com.br/medicamentos/cardiol...,cardiologicos
4,3,Remédio para Arritmia Cardíaca,Remédios Cardiológicos,Medicamentos > Remédios Cardiológicos > Remédi...,https://www.araujo.com.br/medicamentos/arritmi...,arritmia-cardiaca
5,3,Remédio para Insuficiência Cardíaca,Remédios Cardiológicos,Medicamentos > Remédios Cardiológicos > Remédi...,https://www.araujo.com.br/medicamentos/insufic...,insuficiencia-cardiaca
6,3,Remédio para Colesterol,Remédios Cardiológicos,Medicamentos > Remédios Cardiológicos > Remédi...,https://www.araujo.com.br/medicamentos/colesterol,colesterol
7,3,Remédio para Pressão Alta,Remédios Cardiológicos,Medicamentos > Remédios Cardiológicos > Remédi...,https://www.araujo.com.br/medicamentos/hiperte...,hipertensao
8,2,Mais Medicamentos,Medicamentos,Medicamentos > Mais Medicamentos,https://www.araujo.com.br/medicamentos/mais-me...,mais-medicamentos
9,2,Antigripais,Medicamentos,Medicamentos > Antigripais,https://www.araujo.com.br/medicamentos/antigripal,antigripal


## 4. Conferindo o resultado

In [4]:
print("Categorias por nível:")
print(df["nivel"].value_counts().sort_index().to_string())

Categorias por nível:
nivel
1     11
2    119
3    620


In [5]:
# os 11 departamentos (nível 1) e quantas subcategorias cada um tem
depto = df[df["nivel"] == 1]["categoria"].tolist()

resumo = (df[df["nivel"] > 1]
          .assign(departamento=lambda d: d["caminho"].str.split(" > ").str[0])
          .groupby("departamento")
          .size()
          .reindex(depto)
          .rename("subcategorias")
          .reset_index())

resumo

,departamento,subcategorias
0,Medicamentos,111
1,Infantil,77
2,Dermocosméticos,28
3,Saúde e Bem Estar,128
4,Beleza e Cuidados,37
5,Higiene Pessoal,43
6,Pet Shop,44
7,Nutrição Saudável,67
8,Mercado,100
9,Maquiagem,30


In [6]:
# árvore de um departamento, para inspeção visual
recorte = df[df["caminho"].str.startswith("Medicamentos")]

for _, linha in recorte.head(30).iterrows():
    print("  " * (linha["nivel"] - 1) + "- " + linha["categoria"])

- Medicamentos
  - Remédio para Diabetes
    - Insulina
  - Remédios Cardiológicos
    - Remédio para Arritmia Cardíaca
    - Remédio para Insuficiência Cardíaca
    - Remédio para Colesterol
    - Remédio para Pressão Alta
  - Mais Medicamentos
  - Antigripais
  - Saúde dos Olhos
    - Colírio Antialérgico
    - Colírio Lubrificante
    - Colírio para Dor e Anti-inflamatório
    - Colírio para Infecções e Irritações
    - Remédio para Glaucoma
    - Produtos para Lente de Contato
  - Saúde dos Ossos
    - Remédio para Artrite
    - Remédio para Gota
    - Remédio para Osteoporose
  - Remédio para Vermes e Parasitas
    - Remédio Antifúngico
    - Remédio para Piolho
  - Remédio para Pele e Mucosa
    - Remédio para Queloide
    - Removedor de Calos e Verrugas
    - Remédio para Assadura
    - Remédio para Afta
    - Remédio para Herpes


## 5. Salvar

In [7]:
df.to_csv("categorias-araujo.csv", index=False, encoding="utf-8-sig")
print("salvo em categorias-araujo.csv")
df.tail()

salvo em categorias-araujo.csv


,nivel,categoria,categoria_pai,caminho,url,slug
745,3,Máquina de Cortar Cabelo,Acessórios para Cabelos,Cabelo > Acessórios para Cabelos > Máquina de ...,https://www.araujo.com.br/cabelo/maquina-de-co...,maquina-de-cortar-cabelo
746,2,Produtos para os Homens,Cabelo,Cabelo > Produtos para os Homens,https://www.araujo.com.br/cabelo/produtos-para...,produtos-para-os-homens
747,3,Shampoo Masculino,Produtos para os Homens,Cabelo > Produtos para os Homens > Shampoo Mas...,https://www.araujo.com.br/cabelo/shampoo-mascu...,shampoo-masculino
748,3,Coloração e Tintura,Produtos para os Homens,Cabelo > Produtos para os Homens > Coloração e...,https://www.araujo.com.br/cabelo/coloracao-e-t...,coloracao-e-tintura-masculinos
749,3,Naturais e Veganos,Produtos para os Homens,Cabelo > Produtos para os Homens > Naturais e ...,https://www.araujo.com.br/cabelo/naturais-e-ve...,naturais-e-veganos-masculinos
